# Amazon Bestsellers: Does Quality Sell, or Does Staying Power Win?

A data story over Amazon's Top 50 bestselling books each year, 2009–2019.

> **The question.** A book's star rating barely predicts how popular it becomes — so what *actually* makes a bestseller last? This notebook follows the data from a counter-intuitive hook (**quality ≠ popularity**) to the real driver of lasting success: **staying power across years.**

All of the loading, cleaning and feature engineering lives in the tested `bestsellers` package under `src/`, so every cell below is one or two lines. The notebook is the story; the module is the engine.

## 1. Setup

Make the `src/` package importable (works whether the kernel starts in the repo root or this `notebooks/` folder), then load a feature-engineered DataFrame and apply the project's shared chart theme.

In [ ]:
import sys
from pathlib import Path

# Walk up from the working directory until we find the package, then put
# src/ on the import path. Robust to where the kernel was launched.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "bestsellers").exists():
        sys.path.insert(0, str(candidate / "src"))
        break

import pandas as pd
import matplotlib.pyplot as plt

from bestsellers.data import load_clean_data, engineer_features
from bestsellers import analysis, viz

viz.set_theme()
pd.set_option("display.precision", 3)

In [ ]:
# Cleaned, analysis-ready data with the derived columns attached.
df = engineer_features(load_clean_data())
df.head()

## 2. Meet the data

One row is a book that charted in a given year — a *book-year*. A title can appear in several years, which is exactly the repetition the staying-power story is built on.

In [ ]:
analysis.dataset_overview(df)

**547 book-years** span **2009–2019**, but only **351 distinct titles** from **248 authors** — so titles and authors recur. Twelve listings are free Kindle promotions, kept on purpose rather than dropped. The gap between 547 chart slots and 351 unique books is the first hint that *staying on the chart* is a thing some books do and most don't.

## 3. The hook — does quality predict popularity?

Intuition says a better-rated book should attract more readers, and therefore more reviews. We can test that directly: correlate each book's **user rating** with its **review count**.

In [ ]:
rating_reviews = analysis.correlation(df, "user_rating", "reviews")
price_reviews = analysis.correlation(df, "price", "reviews")
print(f"rating ~ reviews:  r = {rating_reviews:.3f}")
print(f"price  ~ reviews:  r = {price_reviews:.3f}")

In [ ]:
viz.scatter_rating_vs_reviews(df)
plt.show()

An `r` of essentially **zero**: knowing a book's rating tells you almost nothing about how many reviews it gathered. The cloud is flat. Quality, as captured by stars, and popularity, as captured by reviews, are all but unrelated.

And it isn't just rating — nothing numeric moves together with much strength. Price nudges reviews only slightly (cheaper books pull a few more).

In [ ]:
viz.heatmap_correlations(df)
plt.show()

Every off-diagonal cell sits near zero. So if neither quality nor price explains which books become hits — what does?

## 4. The reveal — staying power

Reframe the question from *how good is a book* to *how long does it last*. The `times_charted` feature counts how many years each title appears in the Top 50.

In [ ]:
analysis.staying_power_summary(df)

In [ ]:
viz.bar_staying_power(df)
plt.show()

Nearly **three quarters of titles chart exactly once** — the long tail of one-hit wonders. A small core endures: a handful reach five years or more, and a single title (the *Publication Manual of the APA*) holds on for all ten.

Now split the books into **one-hit** titles and **repeat bestsellers** (two or more years) and compare them head to head.

In [ ]:
analysis.repeat_vs_one_hit(df)

In [ ]:
viz.bar_repeat_vs_one_hit(df, metric="mean_reviews")
plt.show()

This is the whole argument in one table. Repeat bestsellers have **the same mean rating** as one-hit titles (about 4.6 stars either way) — they are *not* better books by the quality measure. Yet they carry **roughly twice the reviews**. Popularity doesn't come from being rated higher; it compounds from *staying on the chart year after year*. Staying power, not star rating, is what separates a lasting bestseller from a flash in the pan.

## 5. Who endures — the authors behind staying power

If endurance is the prize, who wins it? There are two ways to dominate the chart, and they reward different things. Rank authors by **total chart appearances** (book-years) and you reward *endurance*.

In [ ]:
analysis.top_authors(df, by="appearances", n=10)

In [ ]:
viz.barh_top_authors(df, by="appearances", n=10)
plt.show()

Rank instead by **distinct titles** and you reward *volume* — churning out many different books.

In [ ]:
viz.barh_top_authors(df, by="titles", n=10)
plt.show()

The contrast is the point. **Jeff Kinney** tops both — 12 different *Diary of a Wimpy Kid* titles, each charting once: pure volume. **Suzanne Collins** takes a different route — just a few titles (the *Hunger Games* trilogy) charting again and again across the decade: pure endurance. Both dominate; they get there by opposite means.

## 6. Conclusion

Three findings, following the arc:

1. **Quality ≠ popularity.** A book's star rating is essentially uncorrelated with its review count (`r ≈ 0`). Higher-rated books are not meaningfully more popular, and price barely matters either.
2. **Staying power is the real divide.** 73% of titles chart only once. Repeat bestsellers aren't rated any higher than one-hit wonders, yet they accumulate about **twice the reviews** — popularity compounds from longevity, not from quality.
3. **Two routes to the top.** Authors dominate either by volume (Jeff Kinney's many titles) or by endurance (Suzanne Collins's few titles charting for years).

The takeaway: on this chart, lasting success is less about how good a book is and more about how long it stays in front of readers — staying power wins.